# Overview
> **Chuyển đổi từ `4. Training Utilities/3_Emotion_Reg_CNN.ipynb` sang PyTorch**
- Data Augmentation thời gian thực (`RandomHorizontalFlip`, `RandomShift`) + `EarlyStopping` trên FER-2013.


In [1]:
from pathlib import Path
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# (1) Data & Augmentation Pipeline (tương đương `ImageDataGenerator`)


In [2]:
btl02_prep = Path('../../../BTL02/data/preprocessing')
if (btl02_prep / 'train.npz').exists():
    tr = np.load(btl02_prep / 'train.npz'); va = np.load(btl02_prep / 'valid.npz')
    x_train = (tr['images'][:600].astype('float32') / 255.0).reshape(-1, 1, 48, 48)
    y_train = tr['labels'][:600].astype(np.int64)
    x_test = (va['images'][:200].astype('float32') / 255.0).reshape(-1, 1, 48, 48)
    y_test = va['labels'][:200].astype(np.int64)
else:
    rng = np.random.default_rng(42)
    x_train = rng.uniform(0, 1, (800, 1, 48, 48)).astype('float32'); y_train = rng.integers(0, 7, 800, dtype=np.int64)
    x_test = rng.uniform(0, 1, (200, 1, 48, 48)).astype('float32'); y_test = rng.integers(0, 7, 200, dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.tensor(x_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(x_test), torch.tensor(y_test)), batch_size=64, shuffle=False)

def augment_batch(xb):
    out = xb.clone()
    flip = torch.rand(len(out)) > 0.5
    out[flip] = torch.flip(out[flip], dims=[3])
    sy, sx = int(np.random.randint(-4, 5)), int(np.random.randint(-4, 5))
    return torch.roll(out, shifts=(sy, sx), dims=(2, 3))


# (2) Model & (3) Train with Data Augmentation + EarlyStopping


In [3]:
class Emotion_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=0), nn.ReLU(), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=0), nn.ReLU(),
            nn.Conv2d(128, 218, 3, padding=1), nn.ReLU(), nn.Conv2d(218, 218, 3, padding=0), nn.ReLU(), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(218*9*9, 1024), nn.ReLU(), nn.Dropout(0.5), nn.Linear(1024, 7))
    def forward(self, x): return self.classifier(self.features(x))

model = Emotion_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
best_loss, patience, wait = float('inf'), 3, 0

for epoch in range(1, 5):
    model.train()
    for xb, yb in train_loader:
        xb = augment_batch(xb).to(device); yb = yb.to(device)
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    model.eval()
    va_loss, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            out = model(xb.to(device))
            va_loss += criterion(out, yb.to(device)).item() * len(xb)
            correct += (out.argmax(1).cpu() == yb).sum().item()
    va_loss /= len(x_test)
    print(f'Epoch {epoch:02d} | Augmented Val Loss: {va_loss:.4f} | Val Acc: {correct/len(x_test)*100:.2f}%')


Epoch 01 | Augmented Val Loss: 1.8618 | Val Acc: 17.50%


Epoch 02 | Augmented Val Loss: 1.8637 | Val Acc: 24.00%


Epoch 03 | Augmented Val Loss: 1.8534 | Val Acc: 24.00%


Epoch 04 | Augmented Val Loss: 1.8435 | Val Acc: 24.00%
